Import Cell


In [2]:
import token
import spacy
from spacy import displacy
from IPython.display import display, HTML
import torch

nlp = spacy.load("en_core_web_md") # Medium size model

test_sentences = [
    "The cat sat on the mat.",
    "On the mat, the cat was sitting.",
    "A completely different sentence about something else."
]

Keep punctuation for direct copy detection but remove for semantic/keyword based methods

In [4]:

for sent in test_sentences:
    doc = nlp(sent)
    print(f"Sentence: {sent}")
    print(f"Tokens: {[token.text for token in doc]}")
    print("---")


Sentence: The cat sat on the mat.
Tokens: ['The', 'cat', 'sat', 'on', 'the', 'mat', '.']
---
Sentence: On the mat, the cat was sitting.
Tokens: ['On', 'the', 'mat', ',', 'the', 'cat', 'was', 'sitting', '.']
---
Sentence: A completely different sentence about something else.
Tokens: ['A', 'completely', 'different', 'sentence', 'about', 'something', 'else', '.']
---


In [5]:

class TextPreprocessor:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_md")

    @staticmethod
    def direct_detection(text):
        """For direct copy detection"""
        #Keep punctuation
        return text.lower().strip()

    def semantic_analysis(self, text):
        """Semantic Similarity"""
        doc = self.nlp(text)
        processed_tokens = []
        # Remove stopwords, punctuation
        for token in doc:
            if not token.is_punct and not token.is_space and token.is_alpha and not token.is_stop:
                processed_tokens.append(token.lemma_.lower())
        return " ".join(processed_tokens)

    def syntactic_analysis(self, text):
        """Syntactic Similarity"""
        doc = self.nlp(text)
        processed_tokens = []

        # Normalize content words
        for token in doc:
            if token.is_space:
                continue
            elif token.is_punct:
                processed_tokens.append(token.text) # Keep punctuation
            elif token.is_stop:
                processed_tokens.append(token.lemma_.lower()) # Normalize stopwords
            else:
                processed_tokens.append(token.lemma_.lower()) # Normalize content words
        return " ".join(processed_tokens)


preprocessor = TextPreprocessor()

processed_direct = []
processed_semantic = []
processed_syntactic = []

for sentence in test_sentences:
    print("-" * 50)
    print(f"Sentence: {sentence}")
    direct = preprocessor.direct_detection(sentence)
    processed_direct.append(direct)
    print("--- Direct Sentence ---")
    print(f"{direct}")
    semantic = preprocessor.semantic_analysis(sentence)
    processed_semantic.append(semantic)
    print("--- Semantic Sentence ---")
    print(f"{semantic}")
    syntactic = preprocessor.syntactic_analysis(sentence)
    processed_syntactic.append(syntactic)
    print("--- Syntactic Sentence ---")
    print(f"{syntactic}")

print("-" * 50)
#for sent in test_sentences:
#    print(f"Original Sentence: {sent}")
#    print("--- Semantic Analysis  ---")
#    print(f"Preprocessed Sentence: {preprocessor.semantic_analysis(sent)}")
#    print("--- Syntactic Analysis ---")
#    print(f"Preprocessed Sentence: {preprocessor.syntactic_analysis(sent)}")
#    print("-" * 50)

--------------------------------------------------
Sentence: The cat sat on the mat.
--- Direct Sentence ---
the cat sat on the mat.
--- Semantic Sentence ---
cat sit mat
--- Syntactic Sentence ---
the cat sit on the mat .
--------------------------------------------------
Sentence: On the mat, the cat was sitting.
--- Direct Sentence ---
on the mat, the cat was sitting.
--- Semantic Sentence ---
mat cat sit
--- Syntactic Sentence ---
on the mat , the cat be sit .
--------------------------------------------------
Sentence: A completely different sentence about something else.
--- Direct Sentence ---
a completely different sentence about something else.
--- Semantic Sentence ---
completely different sentence
--- Syntactic Sentence ---
a completely different sentence about something else .
--------------------------------------------------


In [6]:

def extract_parse_tree(text):
    doc = nlp(text)

    print(f"Sentence: {text}")
    print("\nDependenct Parse Tree:")
    print("-" * 50)

    for token in doc:
        print(f"{token.text:<12} {token.dep_:<12} {token.head.text:<12} {[child.text for child in token.children]}")

    return doc

for sentence in processed_syntactic:
    doc = extract_parse_tree(sentence)
    print("\n" + "="*60 + "\n")

Sentence: the cat sit on the mat .

Dependenct Parse Tree:
--------------------------------------------------
the          det          cat          []
cat          nsubj        sit          ['the']
sit          ROOT         sit          ['cat', 'on', '.']
on           prep         sit          ['mat']
the          det          mat          []
mat          pobj         on           ['the']
.            punct        sit          []


Sentence: on the mat , the cat be sit .

Dependenct Parse Tree:
--------------------------------------------------
on           prep         sit          ['mat']
the          det          mat          []
mat          pobj         on           ['the']
,            punct        sit          []
the          det          cat          []
cat          nsubj        sit          ['the']
be           aux          sit          []
sit          ROOT         sit          ['on', ',', 'cat', 'be', '.']
.            punct        sit          []


Sentence: a completely dif

***USE NetworkX

In [7]:


def visualize_parse_tree(text):
    doc = nlp(text)
    html = displacy.render(doc, style="dep", jupyter=False, options={"distance": 100})
    display(HTML(html))



for sentence in processed_syntactic:
    print(f"Sentence: {sentence}")
    print("---")
    print(f"Processed Sentence: " + sentence)
    visualize_parse_tree(sentence)

Sentence: the cat sit on the mat .
---
Processed Sentence: the cat sit on the mat .


Sentence: on the mat , the cat be sit .
---
Processed Sentence: on the mat , the cat be sit .


Sentence: a completely different sentence about something else .
---
Processed Sentence: a completely different sentence about something else .
